In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 549.5 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 34.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.0 MB/s eta 0:00:00


In [4]:
import os
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from mlflow import MlflowClient

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/gormo22/ML-FraudDetection.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gormo22"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
print("MLflow tracking URI set.")

MLflow tracking URI set.


In [6]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection/"

test = pd.read_csv(DATA_DIR + "test_transaction.csv").merge(
    pd.read_csv(DATA_DIR + "test_identity.csv"),
    on="TransactionID",
    how="left"
)

test_ids = test["TransactionID"]

test.drop(columns=["TransactionID"], inplace=True)

print("Test shape:", test.shape)

Test shape: (506691, 432)


In [ ]:
PIPELINE_RUN_ID = "db51fa4aeb9b4858a3f3c82232f17a01"

artifact_path = "pipeline/fraud_pipeline.pkl"

local_path = mlflow.artifacts.download_artifacts(
    run_id=PIPELINE_RUN_ID,
    artifact_path=artifact_path
)

print("Pipeline loaded successfully")

In [15]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection/"

train = pd.read_csv(DATA_DIR + "train_transaction.csv").merge(
    pd.read_csv(DATA_DIR + "train_identity.csv"),
    on="TransactionID",
    how="left"
)

test = pd.read_csv(DATA_DIR + "test_transaction.csv").merge(
    pd.read_csv(DATA_DIR + "test_identity.csv"),
    on="TransactionID",
    how="left"
)

test.columns = test.columns.str.replace('-', '_')

y = train["isFraud"]
train_ids = train["TransactionID"]
test_ids = test["TransactionID"]

train.drop(["TransactionID", "isFraud"], axis=1, inplace=True)
test.drop(["TransactionID"], axis=1, inplace=True)

In [16]:
X_train = pipeline.fit_transform(train, y)
X_test  = pipeline.transform(test)

print(X_train.shape, X_test.shape)

(590540, 92) (506691, 92)


In [20]:
import mlflow
import pandas as pd

XGB_RUN_ID = "5886197206e045caa2c81e065d704f74"

model_uri = f"runs:/{XGB_RUN_ID}/model"

model = mlflow.lightgbm.load_model(model_uri)

print("LGBM model loaded ✔")

LGBM model loaded ✔


In [22]:
print("Making predictions on test data...")
y_pred_proba = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": y_pred_proba
})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Predictions saved to {submission_path} successfully!")
submission.head()

Making predictions on test data...
Predictions saved to submission.csv successfully!


,TransactionID,isFraud
0,3663549,0.002199
1,3663550,0.001335
2,3663551,0.000616
3,3663552,0.000931
4,3663553,0.001033
